# CoralNet ETL output evaluation

Explore production ETL parquet outputs on S3 (audit, annotations, images) using Ibis + DuckDB.

Set `RUN` to the version folder under `s3://dev-datamermaid-sm-sources/dev/`.

## 1. Setup

In [ ]:
import os

if "SM_CURRENT_HOST" not in os.environ:
    os.environ["AWS_PROFILE"] = "mermaid-core"

In [ ]:
import hvplot.pandas  # noqa: F401 — registers .hvplot on pandas
import ibis
import pandas as pd
from great_tables import GT

ibis.options.interactive = True

In [ ]:
con = ibis.duckdb.connect()
con.raw_sql("""
    INSTALL httpfs;
    LOAD httpfs;
""")
con.raw_sql("CREATE OR REPLACE SECRET s3 (TYPE S3, PROVIDER CREDENTIAL_CHAIN)")

In [ ]:
BUCKET = "dev-datamermaid-sm-sources"
RUN = "20260521_bd1ad74"
BASE = f"s3://{BUCKET}/dev/{RUN}"

AUDIT_URI = f"{BASE}/coralnet_audit_{RUN}.parquet"
ANNOTATIONS_URI = f"{BASE}/coralnet_annotations_{RUN}.parquet"
IMAGES_URI = f"{BASE}/coralnet_images_{RUN}.parquet"

## 2. Load ETL outputs

In [ ]:
audit = con.read_parquet(AUDIT_URI)
annotations = con.read_parquet(ANNOTATIONS_URI)
images = con.read_parquet(IMAGES_URI)

In [ ]:
row_counts = pd.DataFrame(
    {
        "dataset": ["audit", "annotations", "images"],
        "rows": [
            audit.count().execute(),
            annotations.count().execute(),
            images.count().execute(),
        ],
    }
)

In [ ]:
GT(row_counts).tab_header(title=f"Row counts ({RUN})")

## 3. Audit summary

In [ ]:
summary_data = {
    "Metric": [
        "Total sources",
        "Complete sources",
        "Sources with images folder",
        "Sources with annotations.csv",
        "Sources with image_list.csv",
        "Sources with labelset.csv",
        "Sources with metadata.csv",
        "Sources with image count match",
        "Sources with empty annotations.csv",
        "Sources with empty image_list.csv",
        "Sources with empty labelset.csv",
        "Sources with empty metadata.csv",
    ],
    "Count": [
        audit.count().execute(),
        audit.filter(audit.is_complete).count().execute(),
        audit.filter(audit.has_images_folder).count().execute(),
        audit.filter(audit.has_annotations_csv).count().execute(),
        audit.filter(audit.has_image_list_csv).count().execute(),
        audit.filter(audit.has_labelset_csv).count().execute(),
        audit.filter(audit.has_metadata_csv).count().execute(),
        audit.filter(audit.image_count_match).count().execute(),
        audit.filter(audit.annotations_empty).count().execute(),
        audit.filter(audit.image_list_empty).count().execute(),
        audit.filter(audit.labelset_empty).count().execute(),
        audit.filter(audit.metadata_empty).count().execute(),
    ],
}
summary_df = pd.DataFrame(summary_data)
total_sources = summary_df.loc[0, "Count"]
summary_df["Percentage"] = (summary_df["Count"] / total_sources * 100).round(1)

In [ ]:
GT(summary_df).tab_header(title="CoralNet source audit summary")

In [ ]:
totals = audit.aggregate(
    total_images_s3=audit.n_images_s3.sum(),
    total_images_csv=audit.n_images_csv.sum(),
    total_annotations=audit.n_annotations.sum(),
    total_unique_annotated=audit.n_unique_images_annotated.sum(),
    total_labels=audit.n_labels.sum(),
    total_metadata_rows=audit.n_metadata_rows.sum(),
).execute()

totals_df = totals.T.reset_index()
totals_df.columns = ["Metric", "Value"]

In [ ]:
GT(totals_df).tab_header(title="Total resource counts (from audit)")

In [ ]:
audit.head()

In [ ]:
incomplete_df = audit.filter(~audit.is_complete).order_by("source_id").execute()

incomplete_df.count()

In [ ]:
incomplete_df.to_parquet("incomplete_df.parquet")

### Incomplete-source remediation



`incomplete_df.parquet` feeds **`coralnet-remediate`** (merge with a production **audit** parquet, HTTP probe, write `reconciliation_report.parquet`), then optional redownload and a **targeted ETL audit**:



```sh

uv run coralnet-remediate probe --incomplete-parquet incomplete_df.parquet --audit-parquet <audit.parquet>

uv run coralnet-remediate redownload --report reconciliation_report.parquet

uv run coralnet-etl audit --source-ids-file incomplete_df.parquet --output-dir ./outputs/coralnet-remediation --workers 16

```



Set `CORALNET_USERNAME` / `CORALNET_PASSWORD` (and `AWS_PROFILE`) first. See `wiki/CoralNet-ETL.md` and `mermaidseg/datasets/coralnet/README.md` for details.



## 4. Annotations

In [ ]:
ann_by_source = (
    annotations.group_by("source_id")
    .aggregate(n_annotations=annotations.count())
    .order_by(ibis.desc("n_annotations"))
    .execute()
)

ann_summary = pd.DataFrame(
    {
        "Metric": [
            "Sources in annotations parquet",
            "Total annotation rows",
            "Median annotations per source",
            "Max annotations per source",
        ],
        "Value": [
            len(ann_by_source),
            int(ann_by_source["n_annotations"].sum()),
            float(ann_by_source["n_annotations"].median()),
            int(ann_by_source["n_annotations"].max()),
        ],
    }
)

In [ ]:
GT(ann_summary).tab_header(title="Annotations overview")

In [ ]:
status_counts = (
    annotations.group_by("status")
    .aggregate(n=annotations.count())
    .order_by(ibis.desc("n"))
    .execute()
)

In [ ]:
GT(status_counts).tab_header(title="Annotation rows by status")

## 5. Images

In [ ]:
header_status_counts = (
    images.group_by("header_status").aggregate(n=images.count()).order_by(ibis.desc("n")).execute()
)

img_summary = pd.DataFrame(
    {
        "Metric": [
            "Unique images",
            "Images needing resize",
            "Images with null width",
        ],
        "Value": [
            images.count().execute(),
            images.filter(images.needs_resize).count().execute(),
            images.filter(images.width.isnull()).count().execute(),  # noqa: PD003
        ],
    }
)

In [ ]:
GT(img_summary).tab_header(title="Images overview")

In [ ]:
GT(header_status_counts).tab_header(title="JPEG header scan status")

## 6. Visualizations

In [ ]:
audit_df = audit.execute()

completeness_counts = (
    audit_df[
        [
            "has_images_folder",
            "has_annotations_csv",
            "has_image_list_csv",
            "has_labelset_csv",
            "has_metadata_csv",
        ]
    ]
    .sum()
    .reset_index()
)
completeness_counts.columns = ["file_type", "count"]
completeness_counts["file_type"] = (
    completeness_counts["file_type"].str.replace("has_", "").str.replace("_", " ")
)

In [ ]:
completeness_counts.hvplot.bar(
    x="file_type",
    y="count",
    title="Sources by file availability",
    xlabel="File type",
    ylabel="Number of sources",
    rot=45,
    height=400,
    width=600,
)

In [ ]:
complete_df = audit_df[audit_df["is_complete"]]
complete_df.hvplot.scatter(
    x="n_images_s3",
    y="n_annotations",
    title="Images vs annotations per source",
    xlabel="Number of images",
    ylabel="Number of annotations",
    hover_cols=["source_id"],
    height=400,
    width=600,
)

In [ ]:
ann_hist_df = ann_by_source

In [ ]:
edge_df = (
    images.filter(images.longest_edge.notnull()).select("longest_edge", "needs_resize").execute()  # noqa: PD004
)

In [ ]:
edge_df.hvplot.hist(
    "longest_edge",
    by="needs_resize",
    bins=50,
    title="Longest edge distribution (by needs_resize)",
    xlabel="Longest edge (px)",
    ylabel="Images",
    height=400,
    width=700,
    legend="top_right",
)

In [ ]:
dimensions_df = (
    images.filter(images.width.notnull() & images.height.notnull())  # noqa: PD004
    .select("source_id", "width", "height", "needs_resize", "header_status")
    .execute()
)

In [ ]:
dimensions_df.hvplot.scatter(
    x="width",
    y="height",
    by="needs_resize",
    title="CoralNet Image Dimensions",
    xlabel="Width (px)",
    ylabel="Height (px)",
    alpha=0.4,
    size=8,
    height=450,
    width=700,
    legend="top_right",
)

In [ ]:
dimensions_df.hvplot.scatter(
    x="width",
    y="height",
    c="source_id",
    cmap="turbo",
    colorbar=True,
    hover_cols=["source_id", "needs_resize", "header_status"],
    title="Image dimensions colored by source_id",
    xlabel="Width (px)",
    ylabel="Height (px)",
    alpha=0.5,
    size=10,
    height=450,
    width=850,
)

In [ ]:
# Per-source dimension statistics
from ibis import _  # noqa: E402

# Per-source dimension statistics
source_dims = (
    images.group_by("source_id")
    .aggregate(
        n_images=_.count(),
        mean_width=_.width.mean(),
        std_width=_.width.std(),
        cv_width=(_.width.std() / _.width.mean()) * 100,
        mean_height=_.height.mean(),
        std_height=_.height.std(),
        cv_height=(_.height.std() / _.height.mean()) * 100,
        mean_aspect_ratio=(_.width / _.height).mean(),
        min_width=_.width.min(),
        max_width=_.width.max(),
        min_height=_.height.min(),
        max_height=_.height.max(),
    )
    .order_by(ibis.desc("n_images"))
    .execute()
)

In [ ]:
source_dims  # noqa: B018

In [ ]:
# Show sources with highest/lowest consistency (CV)
consistency_summary = source_dims[
    [
        "source_id",
        "n_images",
        "mean_width",
        "cv_width",
        "mean_height",
        "cv_height",
        "mean_aspect_ratio",
    ]
].head(20)

GT(consistency_summary).tab_header(
    title="Per-source dimension consistency (top 20 by image count)",
    subtitle="CV = coefficient of variation (lower = more consistent)",
)

In [ ]:
# Scatter: CV width vs CV height (each point is a source)
source_dims.hvplot.scatter(
    x="cv_width",
    y="cv_height",
    size="n_images",
    scale=0.5,
    hover_cols=["source_id", "n_images"],
    title="Source consistency: width CV vs height CV (bubble size = # images)",
    xlabel="Width CV (%)",
    ylabel="Height CV (%)",
    alpha=0.6,
    height=450,
    width=700,
)

In [ ]:
# For box plots, filter to sources with enough images and add source labels
top_sources = source_dims.nlargest(12, "n_images")["source_id"].tolist()
boxplot_df = dimensions_df[dimensions_df["source_id"].isin(top_sources)].copy()
boxplot_df["source"] = "s" + boxplot_df["source_id"].astype(str)

In [ ]:
boxplot_df.hvplot.box(
    y="width",
    by="source",
    title="Width distribution by source (top 12)",
    ylabel="Width (px)",
    xlabel="Source",
    height=400,
    width=900,
    rot=45,
)

In [ ]:
boxplot_df.hvplot.box(
    y="height",
    by="source",
    title="Height distribution by source (top 12)",
    ylabel="Height (px)",
    xlabel="Source",
    height=400,
    width=900,
    rot=45,
)

In [ ]:
images.head()